### **Fact Orders**

**Data Reading**

In [0]:
df = spark.sql("select * from  databricksete_cat.silver.orders_silver")
display(df)

In [0]:
df_dimcus = spark.sql("select DimCustomerKey, customer_id as dim_customer_id from databricksete_cat.gold.dimcustomers")

df_dimpro = spark.sql("select product_id as DimProductKey, product_id as dim_product_id from databricksete_cat.gold.dimproducts")

**Fact Dataframe**

In [0]:
df_fact = df.join(df_dimcus, df.customer_id == df_dimcus.dim_customer_id, 'left').join(df_dimpro, df.product_id == df_dimpro.dim_product_id, 'left')

df_fact_new = df_fact.drop('dim_customer_id','dim_product_id','customer_id','product_id')

display(df_fact_new)

**Upsert on Fact Table**

In [0]:
from delta.tables import DeltaTable

if spark.catalog.tableExists("databricksete_cat.gold.fact_orders"):
  dlt_obj = DeltaTable.forName(spark, "databricksete_cat.gold.fact_orders")
  dlt_obj.alias("t").merge(
    df_fact_new.alias("s"),
    "t.order_id = s.order_id AND t.DimCustomerKey = s.DimCustomerKey AND t.DimProductKey = s.DimProductKey")\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()
else:
    df_fact_new.write.format("delta")\
        .option("path", "abfss://gold@tjdatabricksete.dfs.core.windows.net/FactOrders")\
        .saveAsTable("databricksete_cat.gold.fact_orders")

In [0]:
%sql
select * from databricksete_cat.gold.fact_orders